# The Yinzer Index - Safety Metric (Darsh)

This notebook calculates a safety score for Pittsburgh neighborhoods using arrest counts.

Idea: fewer arrests means safer neighborhoods for young adults.

In [ ]:
# Step 1: Import libraries and set plotting style
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('ggplot')

## Step 2: Load and inspect arrest data

In [ ]:
# Read the arrest CSV
arrests = pd.read_csv('data/arrest_data.csv')

print('Rows:', len(arrests))
print(arrests.columns.tolist())
arrests.head()

## Step 3: Clean neighborhood column

In [ ]:
# Keep only rows that have a neighborhood name
arrests = arrests.dropna(subset=['INCIDENTNEIGHBORHOOD']).copy()

# Remove extra spaces and drop empty text values
arrests['INCIDENTNEIGHBORHOOD'] = arrests['INCIDENTNEIGHBORHOOD'].astype(str).str.strip()
arrests = arrests[arrests['INCIDENTNEIGHBORHOOD'] != '']

print('Rows after cleaning:', len(arrests))

## Step 4: Count arrests by neighborhood

In [ ]:
# Count number of arrests in each neighborhood
arrest_counts = arrests.groupby('INCIDENTNEIGHBORHOOD').size().reset_index(name='arrest_count')
arrest_counts = arrest_counts.rename(columns={'INCIDENTNEIGHBORHOOD': 'neighborhood'})

arrest_counts.head()

## Step 5: Normalize and invert for safety score

In [ ]:
# Make a 0 to 1 safety score from arrest counts
# Lower arrests should give a higher safety score
lowest = arrest_counts['arrest_count'].min()
highest = arrest_counts['arrest_count'].max()
spread = highest - lowest

if spread == 0:
    arrest_counts['safety_score'] = 1.0
else:
    arrest_counts['safety_score'] = 1 - ((arrest_counts['arrest_count'] - lowest) / spread)

safety_scores = arrest_counts[['neighborhood', 'safety_score']].sort_values('safety_score', ascending=False)
safety_scores.head(10)

## Step 6: Visualize safest and least safe neighborhoods

In [ ]:
# Top 15 safest neighborhoods
safest_15 = safety_scores.head(15).sort_values(by='safety_score', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(safest_15['neighborhood'], safest_15['safety_score'], color='mediumseagreen')
plt.xlabel('Safety Score (0 to 1)')
plt.ylabel('Neighborhood')
plt.title('Top 15 Safest Neighborhoods')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

# Bottom 10 least safe neighborhoods (still shown as horizontal for readability)
least_safe_10 = safety_scores.sort_values(by='safety_score', ascending=True).head(10)
least_safe_10 = least_safe_10.sort_values(by='safety_score', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(least_safe_10['neighborhood'], least_safe_10['safety_score'], color='tomato')
plt.xlabel('Safety Score (0 to 1)')
plt.ylabel('Neighborhood')
plt.title('Bottom 10 Least Safe Neighborhoods')
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

## Step 7: Save scores for final report

In [ ]:
# Save safety score table for merging later
import os
os.makedirs('scores', exist_ok=True)
safety_scores.to_csv('scores/safety_scores.csv', index=False)
print('Saved scores/safety_scores.csv')

## Conclusion

[DARSH - Write 3-4 sentences about the safest and least safe neighborhoods and what that means for young adults.]